# 03 — Statistical tests

Full ANOVA decomposition + correlation analysis.

In [ ]:
import pandas as pd
import sys
sys.path.insert(0, '..')
from analysis.anova_decomposition import run_anova_pingouin, add_size_bins, add_thermal_bins
from analysis.correlation_analysis import compute_ratios, pearson_subgroup

master = pd.read_csv('data/processed/master.csv')
master = add_size_bins(master, [0, 4, 12, 100])
master = add_thermal_bins(master, [0, 38, 42, 55])
print(master.shape)

In [ ]:
# ANOVA on ANE condition
df_ane = master[master['compute_unit'] == 'cpuAndNeuralEngine'].copy()
try:
    import pingouin as pg
    aov = pg.anova(data=df_ane.dropna(subset=['precision','model_size_bin','median_us']),
                   dv='median_us', between=['precision','model_size_bin'], detailed=True)
    print(aov[['Source','F','p-unc','np2']].to_string())
except ImportError:
    print('pip install pingouin')

In [ ]:
# Bandwidth-speedup correlation
ratios = compute_ratios(master)
for prec in ['fp16', 'int8_linear']:
    res = pearson_subgroup(ratios, f'bw_ratio_{prec}', f'speedup_{prec}', 'model_size_mb', 10.0)
    print(f'\n{prec}:')
    for key in ['full', 'cache_resident', 'cache_evicting']:
        s = res.get(key, {})
        print(f'  {key:20s}  r={s.get("r")}  p={s.get("p")}  n={s.get("n")}')

In [ ]:
# Interpretation
print('\nInterpretation key:')
print('  r(cache_evicting) > r(cache_resident) for INT8 → memory-bandwidth-bound → cache effect likely')
print('  Similar r values across subgroups → arithmetic effect dominates → conventional narrative')